In [ ]:
import torch
import warnings

# Selección dinámica de dispositivo
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    warnings.warn("No se detectó un acelerador (GPU/MPS). La ejecución será lenta.")

print(f"Usando dispositivo: {device}")


```{admonition} Ejecutar en Google Colab
:class: tip

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/salvahin/ACA-2026/blob/main/book/notebooks/02_cuda_pytorch_internals.ipynb)
```


In [ ]:
# Setup Colab Environment
!pip install -q numpy pandas matplotlib seaborn scikit-learn torch transformers accelerate triton xgrammar
print('Dependencies installed!')

In [ ]:
import torch

# Crear tensor en CPU
x_cpu = torch.randn(1000, 1000)

# Mover a GPU
x_gpu = x_cpu.to('cuda')        # Copia explícita
x_gpu = x_cpu.cuda()            # Equivalente

# Crear directamente en GPU
x_gpu = torch.randn(1000, 1000, device='cuda')

# Verificar dispositivo
print(x_gpu.device)  # cuda:0

### Costo del Movimiento CPU↔GPU

```{admonition} 🧠 Modelo Mental: CPU↔GPU Transfer
:class: hint
Piensa en CPU y GPU como dos ciudades conectadas por un puente:
- **Dentro de GPU**: Autopista de 12 carriles (2000 GB/s HBM)
- **CPU↔GPU**: Puente angosto (32 GB/s PCIe)

Mover 1GB de CPU a GPU = ~30ms (¡es MUCHO tiempo en GPU!)
Durante esos 30ms, tu GPU está parada esperando datos.
```

```
PCIe Gen4: ~32 GB/s bidireccional
HBM (GPU interna): ~2000 GB/s

Mover 1GB de CPU a GPU:
- Tiempo: ~30ms
- Durante este tiempo, la GPU está esperando

Regla: Minimiza transferencias CPU↔GPU
```

```{admonition} ⚡ Tip de Performance: Minimizar Transferencias
:class: important
Para evitar cuello de botella PCIe:
1. **Crea tensores directamente en GPU**: `torch.randn(..., device='cuda')`
2. **Batch múltiples operaciones**: No muevas datos después de cada operación
3. **Usa pinned memory** para transferencias asíncronas cuando sea necesario
4. **Pipeline**: Overlap transferencias con compute usando streams
```

```{admonition} Concepto clave
:class: note
El bandwidth PCIe es 60x menor que el HBM interno. Mantén los datos en GPU el mayor tiempo posible. Una regla simple: si tu kernel tarda <30ms pero mueves 1GB de datos, la transferencia es tu bottleneck real.
```

### Memory Layout y Contiguidad

In [ ]:
# Tensor contiguo (óptimo para GPU)
x = torch.randn(100, 100)
print(x.is_contiguous())  # True

# Tensor no contiguo (después de transpose)
y = x.t()
print(y.is_contiguous())  # False

# Hacer contiguo (copia los datos)
y_contig = y.contiguous()

Tensores no contiguos causan accesos de memoria ineficientes.

### Caching Allocator de PyTorch

PyTorch usa un **caching allocator** para evitar llamadas frecuentes a `cudaMalloc`:

In [ ]:
# Primera allocación: llama a cudaMalloc
x = torch.randn(1000, 1000, device='cuda')

# Liberar tensor
del x

# Segunda allocación del mismo tamaño: reutiliza memoria cacheada
y = torch.randn(1000, 1000, device='cuda')  # No llama cudaMalloc

### Monitoreo de Memoria

In [ ]:
# Memoria actualmente allocada
allocated = torch.cuda.memory_allocated()

# Memoria reservada por el caching allocator
reserved = torch.cuda.memory_reserved()

# Pico de memoria
max_allocated = torch.cuda.max_memory_allocated()

print(f"Allocated: {allocated / 1e9:.2f} GB")
print(f"Reserved:  {reserved / 1e9:.2f} GB")
print(f"Peak:      {max_allocated / 1e9:.2f} GB")

# Liberar memoria cacheada
torch.cuda.empty_cache()

### CUDA Streams

Un **CUDA stream** es una secuencia de operaciones que se ejecutan en orden. Streams diferentes pueden ejecutarse en paralelo.

In [ ]:
# Stream por defecto
x = torch.randn(1000, 1000, device='cuda')
y = x * 2  # Ejecuta en stream por defecto

# Crear stream custom
stream = torch.cuda.Stream()

with torch.cuda.stream(stream):
    z = x + y
    w = z * 3

# Sincronización
torch.cuda.synchronize()  # Todos los streams
stream.synchronize()       # Stream específico

### Paralelismo con Streams

In [ ]:
# Sin streams: secuencial
def sequential():
    for i in range(10):
        x = torch.randn(1000, 1000, device='cuda')
        y = x @ x.t()
    torch.cuda.synchronize()

# Con streams: paralelo
def parallel():
    streams = [torch.cuda.Stream() for _ in range(10)]
    results = []

    for i, stream in enumerate(streams):
        with torch.cuda.stream(stream):
            x = torch.randn(1000, 1000, device='cuda')
            y = x @ x.t()
            results.append(y)

    torch.cuda.synchronize()
    return results

### torch.profiler

In [ ]:
from torch.profiler import profile, ProfilerActivity

model = torch.nn.Linear(1000, 1000).cuda()
x = torch.randn(100, 1000, device='cuda')

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    for _ in range(10):
        y = model(x)
        y.sum().backward()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

**Output típico:**
```
Name                    CPU Time  CUDA Time  # Calls   Memory
aten::addmm              100us     800us       10      4MB
aten::mm                  80us     600us       10      4MB
aten::sum                 20us     100us       10      0B
```

### Patrones Problemáticos

In [ ]:
# MALO: Muchas operaciones pequeñas
for i in range(1000):
    x = x + 1  # 1000 kernel launches

# BUENO: Una operación grande
x = x + 1000  # 1 kernel launch

# MALO: Sincronización innecesaria
for i in range(100):
    x = model(x)
    print(x[0, 0].item())  # .item() sincroniza!

# BUENO: Batch las sincronizaciones
results = []
for i in range(100):
    x = model(x)
    results.append(x[0, 0])
torch.cuda.synchronize()
print([r.item() for r in results])

### Pinned Memory para Transferencias Rápidas

In [ ]:
# Sin pinned memory
x_cpu = torch.randn(1000, 1000)
x_gpu = x_cpu.cuda()  # Copia síncrona lenta

# Con pinned memory
x_pinned = torch.randn(1000, 1000).pin_memory()
x_gpu = x_pinned.cuda(non_blocking=True)  # Copia asíncrona rápida

---

## Resumen de Conceptos Clave

```{admonition} Resumen
:class: important
**Mapa conceptual CUDA ↔ PyTorch:**

| Concepto | CUDA | PyTorch |
|----------|------|---------|
| Identificar thread | `blockIdx`, `threadIdx` | N/A (abstracción) |
| Lanzar kernel | `<<<bloques, threads>>>` | Automático |
| Sincronizar | `__syncthreads()` | `torch.cuda.synchronize()` |
| Memoria GPU | `cudaMalloc/cudaFree` | Caching allocator |
| Streams | `cudaStream_t` | `torch.cuda.Stream()` |
| Profiling | `nvprof`, `nsight` | `torch.profiler` |

**Checklist de optimización:**
- [ ] Indexación correcta: `idx = blockIdx.x * blockDim.x + threadIdx.x`
- [ ] Coalescing: Threads consecutivos acceden direcciones consecutivas
- [ ] Minimizar CPU↔GPU: Crear tensores en GPU desde el inicio
- [ ] Sincronización: Llamar `torch.cuda.synchronize()` antes de medir tiempo
- [ ] Pinned memory para transferencias asíncronas cuando sea necesario
```

```{admonition} ✅ Verifica tu comprensión
:class: note
**Pregunta 1**: Para grid de 100 bloques con 256 threads/bloque, ¿cuál es el índice global del thread con blockIdx=5, threadIdx=32?
<details><summary>Respuesta</summary>
`idx = 5 * 256 + 32 = 1280 + 32 = 1312`
</details>

**Pregunta 2**: ¿Por qué `x.item()` en un loop es lento?
<details><summary>Respuesta</summary>
Porque `.item()` sincroniza GPU→CPU para cada iteración, bloqueando ejecución asíncrona y pagando overhead de PCIe cada vez.
</details>

**Pregunta 3**: Tienes 1000 operaciones pequeñas. ¿Mejor lanzar 1000 kernels o fusionarlas?
<details><summary>Respuesta</summary>
Fusionar. Cada kernel launch tiene overhead (~5-10μs). 1000 launches = 5-10ms de overhead puro. Un kernel fusionado elimina ese overhead.
</details>
```

---

## Ejercicios y Reflexión

### Ejercicio 1: Cálculo de Índices

Para un grid de 5x3 bloques con 16x16 threads por bloque:
- ¿Cuántos threads totales?
- ¿Cuál es el índice global del thread en bloque (2,1) con threadIdx (3,5)?

### Ejercicio 2: Coalescing

Analiza: ¿Cuál versión tiene mejor coalescing?

```cuda
// Versión A
int idx = threadIdx.x + i;
datos[idx] = procesar(datos[idx]);

// Versión B
int idx = threadIdx.x + blockIdx.x * blockDim.x + i;
datos[idx] = procesar(datos[idx]);
```

### Ejercicio 3: Profiling PyTorch

Usa `torch.profiler` para medir:
1. Multiplicación de matrices 1000x1000
2. Forward pass de ResNet-18

¿Cuál tiene mejor utilización de GPU?

### Para Pensar

> *Si el profiler muestra 80% del tiempo en transferencias CPU↔GPU, ¿qué cambios arquitectónicos considerarías?*

---

## Próximos Pasos

En la siguiente semana, exploraremos **Optimización de Memoria GPU**: coalesced access patterns, bank conflicts, tiling, y el modelo roofline.

---

*Esta lectura es parte del curso "Grammar-Constrained GPU Kernel Generation" - ACA*